# job_yolo11 via SDK — Ultralytics YOLO11m on COCO

Mirrors `../job_yolo11/`. Submits federated fine-tuning of YOLO11m using the Ultralytics framework.

## Recipe (matches `job_yolo11/configs/`)

| Knob | Value |
|---|---|
| `model` | `yolo11m.pt` |
| `imgsz` | 640 |
| `learning_rate` | 0.01 |
| `batch_size` | 16 |
| `local_epochs` | 20 |
| `num_rounds` | 5 |

In [ ]:
# ── Connect to ResonTech ─────────────────────────────────────────────────────
# Reads credentials + endpoints from notebooks/.env (see .env.example).
# pip install python-dotenv resontech
import os

from dotenv import load_dotenv
load_dotenv()

from resontech import (
    ResonTech, ResonTechConfig,
    TrainingConfig, FederationConfig, ModelConfig,
)

_kwargs = dict(
    base_url=os.environ.get("RESONTECH_API",  "https://api.reson.tech"),
    email=os.environ["RESONTECH_EMAIL"],
    password=os.environ["RESONTECH_PASSWORD"],
    s3_access_key_id=os.environ["RESONTECH_S3_KEY"],
    s3_secret_access_key=os.environ["RESONTECH_S3_SECRET"],
)
# Optional URL overrides (for beta / self-hosted deployments)
if "RESONTECH_S3_ENDPOINT" in os.environ:
    _kwargs["s3_endpoint"] = os.environ["RESONTECH_S3_ENDPOINT"]
if "RESONTECH_DASHBOARD" in os.environ:
    _kwargs["dashboard_url"] = os.environ["RESONTECH_DASHBOARD"]
if "RESONTECH_S3_REGION" in os.environ:
    _kwargs["s3_region"] = os.environ["RESONTECH_S3_REGION"]
if "RESONTECH_S3_BUCKET" in os.environ:
    _kwargs["s3_bucket_alias"] = os.environ["RESONTECH_S3_BUCKET"]

config = ResonTechConfig(**_kwargs)
sdk = ResonTech(config)
sdk.login()

In [ ]:
"""Mirrors job_yolo11/scripts/{model_def.py, yolo_utils.py}."""
import os

import torch
import torch.nn as nn
import yaml as _yaml

_orig_load = torch.load
torch.load = lambda *a, **kw: _orig_load(*a, **{"weights_only": False, **kw})

from ultralytics import YOLO


PRETRAINED_ID = "yolo11m.pt"


def _runtime_data_yaml(shard_yaml: str, data_root: str, out_dir: str) -> str:
    with open(shard_yaml) as f:
        data = _yaml.safe_load(f)
    data["path"] = data_root
    runtime = os.path.join(out_dir, "data.yaml")
    with open(runtime, "w") as f:
        _yaml.safe_dump(data, f)
    return runtime


class Model(nn.Module):
    def __init__(self, pretrained_id: str = PRETRAINED_ID):
        super().__init__()
        self._yolo = YOLO(pretrained_id)
        self.inner = self._yolo.model

    def forward(self, x):
        return self.inner(x)


def fl_train(model, env, out_dir, logger=None):
    data_root = env["DATA_ROOT"]
    epochs    = int(env["EPOCHS"])
    batch     = int(env["BATCH_SIZE"])
    imgsz     = int(env.get("IMG_SIZE", 640))
    lr        = float(env["LR"])

    shard_yaml = os.path.join(data_root, "data.yaml")
    runtime_yaml = _runtime_data_yaml(shard_yaml, data_root, out_dir)

    yolo = model._yolo
    yolo.train(
        data=runtime_yaml,
        epochs=epochs,
        batch=batch,
        imgsz=imgsz,
        lr0=lr,
        project=out_dir,
        name="run",
        exist_ok=True,
        amp=True,
    )

    sd = {f"inner.{k}": v.detach().cpu() for k, v in yolo.model.state_dict().items()}
    n_train = sum(
        1 for p in os.scandir(os.path.join(data_root, "images", "train"))
        if p.is_file()
    )
    return {"weights": sd, "samples": n_train}

In [ ]:
training = TrainingConfig(
    local_epochs=20,
    batch_size=16,
    learning_rate=0.01,
    extra={"IMG_SIZE": 640},
)

federation = FederationConfig(num_rounds=5, min_clients=1)

model_config = ModelConfig(model_args={"pretrained_id": "yolo11m.pt"})

job = sdk.rt_submit(
    model=Model,
    name="job_yolo11",
    shards_dir="../job_yolo11/shards",
    requirements_txt="../job_yolo11/requirements/requirements.txt",
    training=training,
    federation=federation,
    model_config=model_config,
)

print(f"Submitted: {job.id}")
print(f"Track at:  {job.dashboard_url}")

## After submission

Open `job.dashboard_url` to follow logs and download the final checkpoint when training finishes. The platform writes the aggregated model to `result.pt` in the bucket and surfaces a download button on the job page.